<h1>Chapter 4 - Memory</h1>
<i>Exploring methodologies for remembering conversations</i>


<a href="https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents/blob/main/chapter04/chapter04_long_term_memory.ipynb)

---

This notebook is for Chapter 4 of [An Illustrated Guide to AI Agents](https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ) by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [1]:
# %%capture
!pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use:

In [2]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 3 12B (no native thinking or tool calling)
llm = LLM(model="gemma3:12b")

If you want to use another LLM, here are a couple of options (both locally and on the cloud) that you can try:

In [3]:
# # llama.cpp
# llm = LLM(model="gemma-3-12B-it-Q4_K_M", base_url="http://127.0.0.1:8080")

# # LM Studio
# llm = LLM(model="gemma-3-12b-it", base_url="http://127.0.0.1:1234/v1")

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-3-12b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

## 2 - Adding Retrieval Augmented Generation (RAG)


In the previous notebook, we added several forms of short-term memory (trimming and summarization). In this notebook we will cover Retrieval Augemented Generation (RAG) as a form of long-term memory.

We start by creating the EmbeddingModel class which uses the OpenAI-endpoint to query a given model. In our example, we are going to be using [EmbeddingGemma](https://ai.google.dev/gemma/docs/embeddinggemma) an embedding model with 308 million parameters. Quite a bit smaller than the 12 billion parameter LLM we have been using thus far!


We then initialize this class and we can use the same client since we are going to be using Ollama as our backend. 

In [4]:
import json
import urllib.request

class EmbeddingModel:
    """Generate embeddings."""

    def __init__(
        self,
        model: str,
        base_url: str = "http://localhost:11434/v1",
    ):
        """Initialize the embedding model with the given model."""
        self.model = model
        self.base_url = base_url

    def embed(self, text: str) -> list[float]:
        """Convert text into a numerical vector."""
        # POST to the OpenAI-compatible /embeddings endpoint
        request = urllib.request.Request(
            f"{self.base_url}/embeddings",
            data=json.dumps({"model": self.model, "input": text}).encode(),
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(request) as resp:
            response = json.loads(resp.read())

        # Extract and return the embedding
        return response["data"][0]["embedding"]

In [5]:
# Initialize EmbeddingGemma
embedding_model = EmbeddingModel(model="embeddinggemma")

# Test the embedding model
output = embedding_model.embed("Dolphins are amazing!")
print(output)
print()
print(len(output))

[-0.21758094, -0.015669592, 0.035899073, -0.004941199, -0.04250471, -0.03879906, -0.027220884, 0.035341103, 0.03916423, -0.028417295, -0.021527335, -0.03848164, -0.037326824, -0.013140361, 0.06105985, 0.03191601, 0.015646826, -0.079750665, -0.04075296, -0.025500597, 0.041090928, 0.029921638, -0.008716447, -0.00071432395, 0.051558282, 0.011706169, -0.02802871, 0.047024127, 0.0019810593, 0.02494449, 0.00044218864, -0.0017245316, 0.013092664, 0.009085177, -0.0029597396, 0.029525086, -0.059515726, -0.032069728, 0.09133975, -0.02486037, -0.03461116, 0.05806378, -0.016218588, -0.019754255, -0.040203847, 0.023236243, -0.02041015, 0.042891268, -0.008285552, -0.006069146, -0.0040035015, -0.061415058, -0.055924766, 0.009008553, -0.020303065, -0.036575016, -0.015122974, 0.009471665, -0.03698199, 0.036770836, -0.019608693, -0.05960811, -0.009101306, 0.005087474, 0.048601866, -0.0615791, 0.0025158275, -0.005496041, 6.5648404e-05, 0.23294322, -0.03692778, -0.02075306, -0.017105525, 0.021757027, 0.13

The output is a list of 768 values, each between -1 and 1.

We can use these values to compare different documents and calculate their similarity. This is typically calculated as the cosine similarity, which represents the angle between embeddings. A smaller angle means a higher similarity. The cosine similarity is calculated through the dot product of the embeddings and then divided by the product of their lengths for normalization.

Let's try it out!

In [6]:
from rich import print

# Create embeddings
embedding_a = embedding_model.embed("I love flamingos.")
embedding_b = embedding_model.embed("Dolphins use echolocation.")
embedding_c = embedding_model.embed("Flamingos are pink birds.")

# Calculate cosine similarity between A and B
dot_ab = sum(x * y for x, y in zip(embedding_a, embedding_b))
norm_a = sum(x * x for x in embedding_a) ** 0.5
norm_b = sum(x * x for x in embedding_b) ** 0.5
similarity_ab = dot_ab / (norm_a * norm_b)

# Calculate cosine similarity between A and C
dot_ac = sum(x * y for x, y in zip(embedding_a, embedding_c))
norm_c = sum(x * x for x in embedding_c) ** 0.5
similarity_ac = dot_ac / (norm_a * norm_c)

print(f"Similarity between [bold]'I love flamingos.'[/bold] and [bold]'Dolphins use echolocation.'[/bold]: {similarity_ab}")
print(f"Similarity between [bold]'I love flamingos.'[/bold] and [bold]'Flamingos are pink birds.'[/bold]: {similarity_ac}")

Similarity between 'I love flamingos.' and 'Dolphins use echolocation.': 0.35435623689809304

Similarity between 'I love flamingos.' and 'Flamingos are pink birds.': 0.6387698376602657






As such, the `RAGMemory` that we are going to implement has the following steps:

1) Embed all external documents the Agent has no direct access to (`__init__`)
2) Embed the user's query (`embedding_model.embed(query)`)
3) Compared the embeddings and create a similarity matrix (`.`)
4) Return the documents with the highest similarity
5) Add those documents to the prompt

In [7]:
from illustrated_agents.chapters.ch4 import Memory

class RAGMemory(Memory):
    """Long-term Memory with RAG."""

    def __init__(self, embedding_model: EmbeddingModel, documents: list[str]):
        super().__init__()
        self.embedding_model = embedding_model
        self.documents = documents
        self.embeddings = [embedding_model.embed(doc) for doc in documents]

    def add(self, role: str, content: str, **kwargs) -> None:
        # Augment user queries with retrieved context before storing
        if role == "user":
            context = "\n".join(self.search(content))
            content = f"""Context:
{context}

Question: {content}"""
        super().add(role, content, **kwargs)

    def search(self, query: str) -> list[str]:
        """Return the top-k documents most similar to the query."""
        query_embed = self.embedding_model.embed(query)
        scores = [self._cosine(query_embed, embed) for embed in self.embeddings]
        ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        return [self.documents[index] for index in ranked[:3]]

    def _cosine(self, a: list[float], b: list[float]) -> float:
        """Calculate cosine similarity between two embeddings.."""
        dot = sum(x * y for x, y in zip(a, b))
        norm = (sum(x * x for x in a) ** 0.5) * (sum(x * x for x in b) ** 0.5)
        return dot / norm

Let’s put this to practice, starting with a set of documents that your TinyAgent has no direct access to. This is going to be a simple example, but imagine you have thousands of documents.

In [8]:
from illustrated_agents.chapters.ch4 import TinyAgent

# RAGMemory with external documents
documents = [
    "Sarah works as a marine biologist studying coral reefs.",
    "Sarah lives in Lisbon, Portugal.",
    "Sarah's favorite hobby is rock climbing.",
    "Sarah favorite animals are flamingos.",
    "Sarah speaks fluent Spanish and Portuguese.",
    "Ilse is a software engineer at a renewable energy startup.",
    "Ilse lives in Amsterdam, the Netherlands.",
    "Ilse plays the cello in a local string quartet.",
    "Ilse's favorite author is Brandon Sanderson.",
    "Ilse's favorite animals are dolphins.",
]
memory = RAGMemory(documents=documents, embedding_model=embedding_model)

# Create the Agent and run a query
agent = TinyAgent(llm=llm, memory=memory)
response = agent.run("What is Sarah's favorite animal?")
print(response)

Sarah's favorite animal is flamingos.

In [9]:
print(agent.memory.get_messages())

[
    {
        'role': 'user',
        'content': "Context:\nSarah favorite animals are flamingos.\nSarah's favorite hobby is rock 
climbing.\nIlse's favorite animals are dolphins.\n\nQuestion: What is Sarah's favorite animal?"
    },
    {'role': 'assistant', 'content': "Sarah's favorite animal is flamingos."}
]

Note how it retrieved the top 3 documents as the context? This is both the advantage and disadvantage of RAG. Although it minimizes the context that you have to pass to the model, there is no guarantee that the context will always be good enough. You could, for example, only accept documents that have a minimum degree of similarity rather than simply getting the top 3 irrespective of their absolute scores.

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

In this chapter, we covered how `Memory` could be added to your `TinyAgent`. There are now three main concepts in total (LLM, Memory, and TinyAgent):

In [10]:
from illustrated_agents.chapters.ch4 import what_we_built_lt; what_we_built_lt

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ ├── agent.py                                                                                                    │
│ ├── llm.py        ← Updated (Add `EmbeddingModel` to support long-term memory.)                                 │
│ ├── memory.py     ← Updated (Add `RAGMemory` to the TinyAgent in the form of RAG.)                              │
│ └── trajectory.py                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# What's Next

In the next chapter, we will further augment the LLM with tools. Although it will not run multiple steps, it will give you the very first step in making the LLM truly actionable.